# Compile Open Supernova Catalog

## Library

In [19]:
import json
import glob
from astropy.table import Table, vstack
import numpy as np
from astropy import units as u
import pandas as pd
from scipy import stats
import multiprocessing as mp
import os
import warnings
warnings.filterwarnings('ignore')

In [20]:
#	Plot presetting
import matplotlib.pyplot as plt
import matplotlib as mpl
#
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [21]:
import sys
sys.path.append(os.path.join('..', 'src'))
# from util.helper import *
# from util.sdtpy import *
from helper import makeSpecColors

## Path

In [22]:
PROJECT_ROOT = os.path.join(os.getcwd(), '..')
DATA = os.path.join(PROJECT_ROOT, 'data')

SPECTRA_DATA = os.path.join(DATA, 'Spectra')
SYNPHOT_DATA = os.path.join(DATA, 'Synphot')

RAW_DATA = os.path.join(DATA, 'raw_data')
RAW_OSC_DATA = os.path.join(RAW_DATA, 'OpenSupernovaCatalog')
SPECTRA_OSC_DATA = os.path.join(SPECTRA_DATA, 'OSC')

## Function

In [23]:
def convert_dict2table(spec_dict):
	# spec_dict['data']에서 파장과 플럭스 데이터 추출
	wavelength = [float(item[0]) for item in spec_dict['data']]
	flux = [float(item[1]) for item in spec_dict['data']]

	# Astropy Table 생성
	spectrum_table = Table([wavelength, flux], names=('wavelength', 'flux'))

	# 생성된 테이블 출력
	# print(spectrum_table)
	return spectrum_table

In [24]:
def put_key_value(_data, _outbl, keyword_to_add):
	# _data = data[objname]

	for key0 in keyword_to_add:
		if key0 in list(_data.keys()):
			subdicts = _data[key0]
			# print(key, subdicts)
			for subdict in subdicts:
				for key, value in subdict.items():
					newkey = f"{key0}_{key}"
					# print(newkey)
					# print(newkey, value)
					if type(value) == list:
						value = ",".join(value)
					_outbl[newkey] = value
	return _outbl

## Setting

In [25]:
keyword_to_add = [
	'claimedtype',
	'ebv',
	'host',
	'hostoffsetang',
	'hostoffsetdist',
	'hostra',
	'hostdec',
	'lumdist',
	'ra',
	'dec',
	'redshift',
	'comovingdist',
	]


In [26]:
# path_osc = '../data/OpenSupernovaCatalog'
# path_save = f"{path_osc}/spectra_data"
# if not os.path.exists(path_save):
# 	os.makedirs(path_save)

# Data

## Raw Data

In [27]:
spectra_paths = [
	os.path.join(RAW_OSC_DATA, 'sne-2020-2024-main'),
	os.path.join(RAW_OSC_DATA, 'sne-2015-2019-master'),
	os.path.join(RAW_OSC_DATA, 'sne-2010-2014-master'),
	os.path.join(RAW_OSC_DATA, 'sne-2005-2009-master'),
	os.path.join(RAW_OSC_DATA, 'sne-2000-2004-master'),
	os.path.join(RAW_OSC_DATA, 'sne-1990-1999-master'),
	os.path.join(RAW_OSC_DATA, 'sne-pre-1990-master'),
]

jsonlist = []
for path_osc_test in spectra_paths:
	jsonlist += sorted(glob.glob(os.path.join(path_osc_test, '*.json')))
print(f"{len(jsonlist):_} json files found")

99_677 json files found


- Sample

In [28]:
nn = 0
jsonfile = jsonlist[nn]
with open(jsonfile, 'r') as file:
	data = json.load(file)
data

{'ASASSN-20ao': {'schema': 'https://github.com/astrocatalogs/supernovae/blob/d3ef5fc/SCHEMA.md',
  'name': 'ASASSN-20ao',
  'sources': [{'name': '2016A&A...594A..13P',
    'bibcode': '2016A&A...594A..13P',
    'reference': 'Planck Collaboration et al. (2016)',
    'alias': '1'},
   {'name': 'ATel 13413',
    'url': 'http://www.astronomerstelegram.org/?read=13413',
    'alias': '2'},
   {'name': 'Gaia Photometric Science Alerts',
    'url': 'http://gsaweb.ast.cam.ac.uk/alerts/alertsindex',
    'alias': '3'},
   {'name': 'ASAS-SN Supernovae',
    'url': 'http://www.astronomy.ohio-state.edu/~assassin/sn_list.html',
    'alias': '4'},
   {'name': 'Latest Supernovae',
    'secondary': True,
    'url': 'http://www.rochesterastronomy.org/snimages/snredshiftall.html',
    'alias': '5'},
   {'name': 'The Open Supernova Catalog',
    'bibcode': '2017ApJ...835...64G',
    'reference': 'Guillochon et al. (2017)',
    'secondary': True,
    'url': 'https://sne.space',
    'alias': '6'}],
  'alias':

In [29]:
objname = list(data.keys())[0]
data[objname].keys()

dict_keys(['schema', 'name', 'sources', 'alias', 'claimedtype', 'comovingdist', 'dec', 'discoverdate', 'discoverer', 'host', 'hostdec', 'hostoffsetang', 'hostoffsetdist', 'hostra', 'lumdist', 'maxabsmag', 'maxappmag', 'maxband', 'maxdate', 'maxvisualabsmag', 'maxvisualappmag', 'maxvisualband', 'maxvisualdate', 'ra', 'redshift', 'velocity', 'photometry'])

# Spectrum data?

- Summary Table

## Summarize the table 

In [30]:
_sumtbl = Table()
_sumtbl['filename'] = jsonlist
_sumtbl['objname']= ' '*100
_sumtbl['spectra'] = False
_sumtbl['n_spectra'] = 0

print(f"{len(_sumtbl)} rows")
print(f"keys: {_sumtbl.keys()}")
_sumtbl[:5]

99677 rows
keys: ['filename', 'objname', 'spectra', 'n_spectra']


filename,objname,spectra,n_spectra
str172,str100,bool,int64
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2020-2024-main/ASASSN-20ao.json,,False,0
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2020-2024-main/ASASSN-20bn.json,,False,0
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2020-2024-main/ASASSN-20bx.json,,False,0
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2020-2024-main/ASASSN-20ep.json,,False,0
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2020-2024-main/ASASSN-20es.json,,False,0


In [31]:
objnames = []
ct = 0
for nn, jsonfile in enumerate(jsonlist):
	print(f"{nn+1:>10}/{len(jsonlist)}", end='\r')
	with open(jsonfile, 'r') as file:
		data = json.load(file)
	#	OBJECT
	objname = list(data.keys())[0]
	keys = list(data[objname].keys())
	# for key in keys:
	# 	if "spectra" in key:
	if 'spectra' in keys:
		# print(objname)
		ct += 1
		n_spectra = len(data[objname]['spectra'])
		_sumtbl['spectra'][nn] = True
		_sumtbl['n_spectra'][nn] = n_spectra
		# data_dict[objname] = data
		objnames.append(objname)
		_sumtbl['objname'][nn] = objname

KeyboardInterrupt: 

In [ ]:
sumtbl = _sumtbl[_sumtbl['spectra'] == True]
print(f"Total Object with Spectra: {len(sumtbl):_}/{len(_sumtbl):_} ({len(sumtbl)/len(_sumtbl):.1%})")
print(f"Total Spectra: {np.sum(sumtbl['n_spectra']):_}")

Total Object with Spectra: 6_606/99_677 (6.6%)
Total Spectra: 20_822


## Generate Spectra all files and plot (for each object)

In [ ]:
plot_check = False
# plot_check = True
spectra_save_check = False
# spectra_save_check = True

MIN_7DT_WAVELENGTH = 4000 - 125
MAX_7DT_WAVELENGTH = 8875 + 125

In [ ]:
out_tables = []

os.makedirs(SPECTRA_OSC_DATA, exist_ok=True)

sumtbl['path_spectra_file'] = ' '*1000
for ss, (objname, jsonfile) in enumerate(zip(sumtbl['objname'], sumtbl['filename'])):

	# 로그 출력
	print(f"[{os.path.basename(jsonfile)}, {ss+1:0>4}/{len(sumtbl)} ({(ss+1)/len(sumtbl):.1%})] {objname}"+" "*20, end='\r')

	with open(jsonfile, 'r') as file:
		data = json.load(file)

	spec_dicts = data.get(objname, {}).get('spectra', [])
	if not spec_dicts:
		continue

	# 플롯 조건 확인
	if plot_check:
		plt.close()

	for jj, spec_dict in enumerate(spec_dicts):
		if 'data' not in spec_dict:
			continue

		sptbl = convert_dict2table(spec_dict)

		#	check outlier flux point within 7DT wavelength range
		indx_wavelength = (sptbl['wavelength'] >= MIN_7DT_WAVELENGTH) & (sptbl['wavelength'] <= MAX_7DT_WAVELENGTH)
		flux = sptbl['flux'][indx_wavelength]
		has_nan = np.any(np.isnan(flux))
		has_inf = np.any(np.isinf(flux))
		has_bad = np.any(~np.isfinite(flux))  # NaN, +inf, -inf 모두 포함
		# print("NaN 포함 여부:", has_nan)
		# print("Inf 포함 여부:", has_inf)
		# print("NaN 또는 Inf 포함 여부:", has_bad)


		filename = spec_dict.get('filename', os.path.basename(jsonfile).replace('json', f'{jj:0>2}.ascii'))
		spec_filename = os.path.join(SPECTRA_OSC_DATA, os.path.basename(filename))

		u_fluxes = spec_dict.get('u_fluxes', '?')
		u_wavelengths = spec_dict.get('u_wavelengths', '?')
		flux_unit_flag = u_fluxes == 'erg/s/cm^2/Angstrom'
		sptbl.meta = {k: v for k, v in spec_dict.items() if k != 'data'}

		if plot_check:
			plt.plot(sptbl['wavelength'], sptbl['flux'], lw=1, alpha=0.75, label=f"{jj}")

		_outbl = Table({
			'jsonfile': [jsonfile],
			'specfile': [filename],
			'flux_unit_flag': [flux_unit_flag],

			#	Wavelength
			'wavelength_unit': [u_wavelengths],
			'min_wavelength': [np.min(sptbl['wavelength'])],
			'max_wavelength': [np.max(sptbl['wavelength'])],
			'med_wavelength': [np.median(np.diff(sptbl['wavelength']))],
			'std_wavelength': [np.std(np.diff(sptbl['wavelength']))],
			
			#	Flux
			'flux_unit': [u_fluxes],
			'min_flux': [np.min(sptbl['flux'])],
			'max_flux': [np.max(sptbl['flux'])],
			'med_flux': [np.median(sptbl['flux'])],
			'std_flux': [np.std(sptbl['flux'])],

			#	Quality Check
			'has_nan': [has_nan],
			'has_inf': [has_inf],
			'has_bad': [has_bad],

			#	File Path
			'path_spectra_file': [spec_filename],
		})

		_outbl = put_key_value(_data=data[objname], _outbl=_outbl, keyword_to_add=keyword_to_add)
		out_tables.append(_outbl)

		#	Save the Spectrum
		if not spectra_save_check and not os.path.exists(spec_filename):
			sptbl.write(spec_filename, format='ascii', overwrite=True)

	#	Draw the Plot for each unique object
	plotname = os.path.join(SPECTRA_OSC_DATA, os.path.basename(jsonfile).replace('json', 'png'))
	if plot_check and not os.path.exists(plotname):
		plt.ylabel(f"Flux ({u_fluxes})")
		plt.xlabel(f"Wavelength ({u_wavelengths})")
		plt.title(f"{objname} ({len(_outbl)})")
		plt.legend(ncol=3)
		plt.axvline(x=3750, lw=0.5, zorder=0, alpha=0.5, color='grey',)
		plt.axvline(x=9000, lw=0.5, zorder=0, alpha=0.5, color='grey',)
		# plt.axhline(y=0, lw=0.5, color='k', zorder=0)
		plt.xlim(3000, 10000)
		plt.yscale('log')
		plt.tight_layout()
		plt.savefig(plotname)

print("Done!")

Done!89M.json, 6606/6606 (100.0%)] SN1989M                           0+15441630                          


In [ ]:
outbl = vstack(out_tables)

from astropy.table import MaskedColumn
if isinstance(outbl['claimedtype_value'], MaskedColumn):
	outbl['claimedtype_value'].fill_value = "?"
	outbl['claimedtype_value'] = outbl['claimedtype_value'].filled()

outbl

jsonfile,specfile,flux_unit_flag,wavelength_unit,min_wavelength,max_wavelength,med_wavelength,std_wavelength,flux_unit,min_flux,max_flux,med_flux,std_flux,has_nan,has_inf,has_bad,path_spectra_file,claimedtype_value,claimedtype_source,ebv_value,ebv_derived,ebv_e_value,ebv_source,host_value,host_source,hostoffsetang_value,hostoffsetang_u_value,hostoffsetang_source,hostoffsetdist_value,hostoffsetdist_source,hostra_value,hostra_derived,hostra_u_value,hostra_source,hostdec_value,hostdec_derived,hostdec_u_value,hostdec_source,lumdist_value,lumdist_derived,lumdist_u_value,lumdist_source,ra_value,ra_u_value,ra_source,dec_value,dec_u_value,dec_source,redshift_value,redshift_source,comovingdist_value,comovingdist_derived,comovingdist_u_value,comovingdist_source,hostoffsetang_derived,ra_derived,dec_derived,redshift_kind,redshift_derived,claimedtype_kind,redshift_e_value,claimedtype_probability,lumdist_e_value,lumdist_kind,host_kind,ra_e_value,dec_e_value
str142,str75,bool,str8,float64,float64,float64,float64,str24,float64,float64,float64,float64,bool,bool,bool,str144,str19,str39,str6,bool,str6,str5,str30,str102,str7,str10,str25,str7,str64,str14,bool,str5,str4,str15,bool,str7,str5,str9,bool,str3,str30,str13,str5,str20,str14,str7,str20,str9,str45,str9,bool,str3,str20,bool,bool,bool,str26,bool,str13,str8,str6,str7,str4,str7,str5,str5
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2015-2019-master/ASASSN-15ae.json,ASASSNm15ae_2015-01-11_03:59:23_Asiago.ascii,True,Angstrom,3401.520205,8195.770204,4.732724999999846,4.936445908313681e-07,erg/s/cm^2/Angstrom,-8.099677384e-16,7.453186898e-16,2.7484409930000003e-16,1.7531479036960136e-16,False,False,False,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/ASASSNm15ae_2015-01-11_03:59:23_Asiago.ascii,Ia,"2,3,4,6,7,9,11,13,14",0.027,True,0.0005,"5,12",2MASX J15510599+3425520,"2,7",4.98,arcseconds,"2,7",3.21,"1,2,7,11,12,14",15:51:05.99,True,hours,12,+34:25:52.0,True,degrees,12,141.528,True,Mpc,"1,2,7,11,12",15:51:05.76,hours,"2,6,",+34:25:55.92,degrees,"2,7",0.031238,"2,7,11",137.24,True,Mpc,"1,2,7,11,12",--,--,--,--,--,--,--,--,--,--,--,--,--
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2015-2019-master/ASASSN-15aj.json,ASASSN-15aj_20150215_wifes_B.dat,True,Angstrom,3500.0,5699.90332,0.7744139999999788,0.00015952289912992966,erg/s/cm^2/Angstrom,-1.299447538e-16,7.507507121e-16,2.39725306e-16,1.0522315465812452e-16,False,False,False,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/ASASSN-15aj_20150215_wifes_B.dat,Ia,"3,4,7,8,10,11",0.0656,True,0.0008,"6,9",NGC 3449,"3,7,8",6.36,arcseconds,"3,7",--,--,--,--,--,--,--,--,--,--,48.7362,True,Mpc,"2,3,7,8,9",10:52:53.28,hours,"3,7",-32:55:34.68,degrees,"3,7",0.010921,"3,7,8",48.21,True,Mpc,"2,3,7,8,9",--,--,--,--,--,--,--,--,--,--,--,--,--
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2015-2019-master/ASASSN-15aj.json,ASASSN-15aj_20150215_wifes_R.dat,True,Angstrom,5400.0,9567.015625,1.2524410000005446,0.00034668530511394106,erg/s/cm^2/Angstrom,-1.621932569e-16,7.698474935e-16,3.4402297415e-16,4.746627109932833e-17,False,False,False,/Users/paek/Research/7DT/SED-Classifier/notebook/../data/Spectra/OSC/ASASSN-15aj_20150215_wifes_R.dat,Ia,"3,4,7,8,10,11",0.0656,True,0.0008,"6,9",NGC 3449,"3,7,8",6.36,arcseconds,"3,7",--,--,--,--,--,--,--,--,--,--,48.7362,True,Mpc,"2,3,7,8,9",10:52:53.28,hours,"3,7",-32:55:34.68,degrees,"3,7",0.010921,"3,7,8",48.21,True,Mpc,"2,3,7,8,9",--,--,--,--,--,--,--,--,--,--,--,--,--
/Users/paek/Research/7DT/SED-Classifier/notebook/../data/raw_data/OpenSupernovaCatalog/sne-2015-2019-master/ASASSN-15aj.json,ASASSN-15aj_20150224_wifes_B.dat,True,Angstrom,3500.0,5699.956055,0.7744139999999788,0.00014128312540466042,erg/s/cm^2/Angstrom,-3.066788665e-17,1.070186318e-15,5.004786813e-16,2.1763451906976726e-16,False,False,False,/Users/paek/Research/7DT/SED-Classifier/noteboo

In [ ]:
output_table = os.path.join(SPECTRA_OSC_DATA, 'spectra_info.csv')
outbl.write(output_table, format='csv', overwrite=True)